In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers

In [ ]:
# --- Parâmetros ---
Tamb = 25.0
T0 = 100.0
r_true = 0.005
np.random.seed(42)

In [ ]:
# --- Solução analítica ---
def analytical_solution(t, r):
    return Tamb + (T0 - Tamb) * np.exp(-r * t)

In [ ]:
# --- Dados para a PINN (0–1000s) ---
t_train_pinn = np.linspace(0, 1000, 10).reshape(-1, 1)
T_train_pinn = analytical_solution(t_train_pinn, r_true) + np.random.normal(0, 0.5, size=t_train_pinn.shape)
T_train_pinn[0] = T0

NameError: name 'r' is not defined

In [ ]:

# --- Dados para a regressão (0–300s) ---
t_train_reg = np.linspace(0, 300, 10).reshape(-1, 1)
T_train_reg = analytical_solution(t_train_reg, r_true) + np.random.normal(0, 0.5, size=t_train_reg.shape)
T_train_reg[0] = T0

In [ ]:
# --- Normalizações separadas ---
t_mean_pinn, t_std_pinn = t_train_pinn.mean(), t_train_pinn.std()
t_mean_reg, t_std_reg = t_train_reg.mean(), t_train_reg.std()

t_train_norm_pinn = (t_train_pinn - t_mean_pinn) / t_std_pinn
t_train_norm_reg = (t_train_reg - t_mean_reg) / t_std_reg

In [ ]:
# === PINN ===========
# ====================
log_r = tf.Variable(initial_value=tf.math.log(0.01), dtype=tf.float32, trainable=True)

model_pinn = models.Sequential([
    layers.Input(shape=(1,)),
    layers.Dense(64, activation='tanh'),
    layers.Dense(64, activation='tanh'),
    layers.Dense(1)
])

optimizer = optimizers.Adam(learning_rate=0.001)


In [ ]:
@tf.function
def pinn_loss(t_norm, T_true):
    with tf.GradientTape(persistent=True) as tape:
        tape.watch(t_norm)
        T_pred = model_pinn(t_norm)
        T_pred = tf.squeeze(T_pred, axis=1)
        r = tf.exp(log_r)
        T_t = tape.gradient(T_pred, t_norm) / t_std_pinn
        f = T_t + r * (T_pred - Tamb)
        mse_data = tf.reduce_mean((T_pred - tf.squeeze(T_true))**2)
        mse_phys = tf.reduce_mean(f**2)
        return mse_data + mse_phys

t_tensor_pinn = tf.convert_to_tensor(t_train_norm_pinn, dtype=tf.float32)
T_tensor_pinn = tf.convert_to_tensor(T_train_pinn, dtype=tf.float32)

for epoch in range(3000):
    with tf.GradientTape() as tape:
        loss = pinn_loss(t_tensor_pinn, T_tensor_pinn)
    grads = tape.gradient(loss, model_pinn.trainable_variables + [log_r])
    optimizer.apply_gradients(zip(grads, model_pinn.trainable_variables + [log_r]))



In [ ]:
# Predição com PINN
t_pred = np.linspace(0, 1000, 1000).reshape(-1, 1)
t_pred_norm_pinn = (t_pred - t_mean_pinn) / t_std_pinn
T_pred_pinn = model_pinn.predict(t_pred_norm_pinn, verbose=0)
r_est = tf.exp(log_r).numpy()

In [ ]:
# ============================
# === Regressão simples ======
# ============================
model_reg = models.Sequential([
    layers.Input(shape=(1,)),
    layers.Dense(64, activation='tanh'),
    layers.Dense(64, activation='tanh'),
    layers.Dense(1)
])

model_reg.compile(optimizer=optimizers.Adam(0.001), loss='mse')
model_reg.fit(t_train_norm_reg, T_train_reg, epochs=3000, verbose=0)

In [ ]:
# Predição com regressão (somente até 300s)
t_pred_reg = np.linspace(0, 300, 300).reshape(-1, 1)
t_pred_norm_reg = (t_pred_reg - t_mean_reg) / t_std_reg
T_pred_reg = model_reg.predict(t_pred_norm_reg, verbose=0)

In [ ]:
# Solução analítica
T_exact = analytical_solution(t_pred, r_true)


In [ ]:
# ==========================
# === Gráfico final ========
# ==========================
plt.figure(figsize=(10, 6))
plt.plot(t_pred, T_exact, label='Solução Analítica', color='blue', linewidth=2.5)
plt.plot(t_pred, T_pred_pinn, label=f'PINN (r ≈ {r_est:.5f})', color='green', linewidth=2)
plt.plot(t_pred_reg, T_pred_reg, label='Regressão Simples (0–300s)', color='magenta', linestyle='--', linewidth=2)
plt.scatter(t_train_pinn, T_train_pinn, label='Dados PINN (0–1000s)', color='orange', s=50)
plt.scatter(t_train_reg, T_train_reg, label='Dados Regressão (0–300s)', color='red', marker='x')
plt.axvline(x=300, color='gray', linestyle='--', linewidth=1.2, label='Fim da Regressão')
plt.xlabel('Tempo (s)')
plt.ylabel('Temperatura (°C)')
plt.title('Comparação: PINN (0–1000s) vs Regressão (0–300s)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()